# 🧫🦠 MICOM community modeling (MESB course)

Updated from the [ISB 2020 MICOM course](https://gibbons-lab.github.io/isb_course_2020/micom) for **MICOM ≥ 0.35**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/exercises/GEM3_micom/micom.ipynb)

**Updates vs the 2020 notebook**
- Course data from [MESBcourse](https://github.com/SysBioChalmers/MESBcourse) (`exercises/GEM3_micom/`)
- `plot_association` instead of deprecated `plot_fit`
- `plot_focal_interactions` and `plot_mes` ([viz docs](https://micom-dev.github.io/micom/viz.html))
- `atol=1e-3` (OSQP); `fdr_threshold=0.5` for the small demo cohort
- Legacy LASSO coefficient plot for comparison with `plot_association`

**Public data used (downloaded automatically)**
| Source | What |
|--------|------|
| [MESBcourse/GEM3_micom](https://github.com/SysBioChalmers/MESBcourse/tree/main/exercises/GEM3_micom) | QIIME tables, metadata (from ISB 2020 / Gibbons Lab) |
| [Zenodo 3755182](https://zenodo.org/record/3755182) | AGORA model DB + western diet medium |

Slides: [ISB 2020 MICOM session](https://gibbons-lab.github.io/isb_course_2020/micom)


# 📝 Setup

Course files live in **MESBcourse** (`exercises/GEM3_micom/`). The first code cell clones the repo and sets the working directory.

**To share with students**
- Colab badge above, or:
- `https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/exercises/GEM3_micom/micom.ipynb`

**Runtime:** GPU not required. First full run ~15–30 min (model building + `grow`).


In [ ]:
# MESB course data: QIIME artifacts + metadata
!git clone -q --depth 1 https://github.com/SysBioChalmers/MESBcourse
%cd MESBcourse/exercises/GEM3_micom


## Installation

Run once per Colab session:

In [ ]:
!pip install -q -U "micom>=0.35" biom-format matplotlib scikit-learn

import micom
print("MICOM version:", micom.__version__)


## QIIME 2 artifact support

In [ ]:
!pip install -q numpy Cython biom-format
print("Done!")


# 💻 MICOM

We use the Python interface to MICOM. The same workflows are available in the [QIIME 2 plugin](https://library.qiime2.org/plugins/q2-micom/26/).

![micom overview](https://github.com/micom-dev/q2-micom/raw/706f583a060b91c12c0cec7acea2354fdd0dd320/docs/assets/overview.png)


In [ ]:
from micom.data import test_data

test_data().head()

The `file` column is not required when using a taxonomy database like we will do here. The `id` column specifies identifiers for the taxa and should be expressive and not include spaces or special characters. Each row needs to contain the the abundance of a single taxon in a single sample.

Oh no, that's not what we have generated in the previous step. We only have separate QIIME 2 artifacts 😱

No worries, we can deal with that.

## Importing data from QIIME 2

MICOM can read QIIME 2 artifacts. You don't even need to have QIIME 2 installed for that! But before we do so, let's resolve one issue. We discussed that MICOM summarizes genome-scale models into pangenome-scale models as a first step, but our data are on the ASV level...so how will we know what to summarize?

Basically, specific model database can be used to quickly summarize pangenome-scale models for use within MICOM. So, before we read our data we have to decide which model database to use. We will go with the [AGORA database](https://pubmed.ncbi.nlm.nih.gov/27893703/), which is a curated database of more than 800 bacterial strains that commonly live in the human gut. In particular, we will use a version of this database summarized on the genus rank which can be downloaded from [MICOM data repository](https://doi.org/10.5281/zenodo.3755182) which contains a whole lot of prebuilt databases.



In [ ]:
!wget -q -O agora103_genus.qza https://zenodo.org/record/3755182/files/agora103_genus.qza?download=1

Okay. We've got everything we need now. The data from the prior analysis can be found in the `treasure_chest` folder, so we can use those files.

In [ ]:
from micom.taxonomy import qiime_to_micom

tax = qiime_to_micom(
    "treasure_chest/dada2/table.qza",
    "treasure_chest/taxa.qza",
    collapse_on="genus"
)

Notice the `collapse_on` argument. That will specify the rank on which to sumarize and can be a list of several ranks. When matching taxonomy you can either match by the particular rank of interest (for example, just comparing genus names here), or you could compare the entire taxonomy, which will require all taxonomic ranks prior to the target rank to match. For that you cloud specify `collapse_on=["kingdom", "phylum", "class", "order", "family", "genus"]`.

Taxonomic names will often not match 100% between databases. For instance, the genus name "Prevotella" in one database may be "Prevotella_6" in another. The more ranks you use for matching the more likely are you to run into those issues. However, the more taxonomic ranks you use to match the more confident you can be that your observed taxon really is the same taxon as the one in the model database.

The resulting table will contain the same abundances but it will include more ranks if `collapse_on` is a list. All ranks present in the taxonomy will be used when matching to the database. The GreenGenes database is pretty old and many taxonomic names have been superceded by now. So we will stick with the "lax" option of only matching on genus ranks.

We can also look at the generated MICOM taxonomy.

In [ ]:
tax

One helpful thing to do is to merge in our metadata, so we'll have it at hand for the following steps.

In [ ]:
import pandas as pd

metadata = pd.read_table("metadata.tsv").rename(columns={"id": "sample_id"})
tax = pd.merge(tax, metadata, on="sample_id")
tax

With the taxonomic metadata, we can finally build our community-level models.

## Building community models

With the data we have now, building our models is pretty easy. We just pass our taxonomy table and model database to MICOM. We will remove all taxa that make up less than 2.5% of the community to keep the models small and speed up this tutorial. We will have to specify where to write the models. We will also run that in parallel over two threads. It should take around 10 minutes to finish.

In [ ]:
from micom.workflows import build
from micom import Community
import pandas as pd

manifest = build(tax, "agora103_genus.qza", "models", solver="osqp",
                 cutoff=2.5e-2, threads=2)


For different data a warning may pop up if less than 50% of the abundances can be matched to the database. If this happens, you can still continue, but be aware that such a sparse model may not accurately represent your sample. In lower-biomass 16S amplicon sequencing samples from stool, many reads can match to food components or to host mitochondria and these hits probably do not contribute much to bacterial community metabolism. These hits will be excluded from MICOM.

We won't see any warnings here. So, we will go ahead for now. Let's also take a look what we got back from the `build` process.

In [ ]:
manifest

This will tell you many taxa were found in the database and what fraction of the total abundance was represented by the database. Looks okay here.

So we now have our community models and can leverage MICOM fully by simulating community growth.

## Simulating growth

With our community models built, we can start to simulate growth with the cooperative tradeoff algorithm. Because we have no diet information for our samples, we will apply the same 'average Western Diet' to each individual. We will start by downloading this diet from the [MICOM data repository](https://doi.org/10.5281/zenodo.3755182).

In [ ]:
!wget -q -O western_diet_gut.qza https://zenodo.org/record/3755182/files/western_diet_gut.qza?download=1

This is again a QIIME 2 artifact, which we can load into MICOM.

In [ ]:
from micom.qiime_formats import load_qiime_medium

medium = load_qiime_medium("western_diet_gut.qza")
medium

Many dietary components get absorbed in the small intestine. This medium was created by taking dietary components and depleting all nutrients absorbed in the small intestine by a factor of 10 (indicated by the dilution column).

Okay let's go right ahead and simulate growth. This will take a little while and give us time to dive into some details 🏊

In [ ]:
from micom.workflows import grow
import pickle

growth_results = grow(manifest, "models", medium, tradeoff=0.5, threads=2)

# We'll save the results to a file
pickle.dump(growth_results, open("growth.pickle", "wb"))

If that takes too long or was aborted, we can read it in from the treasure chest.

What kind of results did we get? Well, `grow` returns a tuple of 3 data sets:

1. The predicted growth rate for all taxa in all samples
2. The import and export fluxes for each taxon and the external environment
3. Annotations for the fluxes mapping to other databases

The growth rates are pretty straightforward.

In [ ]:
growth_results.growth_rates.head()

More interesting are the exchange fluxes.

In [ ]:
growth_results.exchanges

So we see how much of each metabolite is either consumed or produced by each taxon in each sample. `tolerance` denotes the accuracy of the solver and tells you the smallest absolute flux that is likely different form zero (i.e. substantial flux). *All of the fluxes are normalized to 1g dry weight of bacteria*. So, you can directly compare fluxes between taxa, even if they are present at very different abundances.

However, the metabolite names may not be very informative. That's why we have our annotations! For instance, to figure out what `ac[e]` is (air conditioning?), we can do the following:

In [ ]:
anns = growth_results.annotations
anns[anns.metabolite == "ac[e]"]

Ohhh, it's acetate. Yeah that makes more sense 🕵️‍♀️. For the AGORA models you can also use the official VMH knowledge base at https://vmh.life maintained by Dr. Thiele's, lab which will give you rich information on metabolites and reactions. For instance, you can find out a lot more about acetate at: https://www.vmh.life/#metabolite/ac.

# 📊 Visualizations

Ok, we have seen that we generate a lot of output data from the growth simulations. But how do we make sense of it all?

We will use the standard visualizations included in MICOM. These tools take in the growth results we obtained before and create visualizations in standalone HTML files that bundle the plots and raw data and can be viewed directly in your browser.

The first things we might want to look at are the growth rates for each taxon.

## Visualization helpers

`clean_growth_results` avoids pandas index/column errors. `lasso_production_associations` reproduces the legacy `plot_fit` LASSO coefficient plot for comparison with `plot_association`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.preprocessing import StandardScaler


def clean_growth_results(gr):
    gr.exchanges = gr.exchanges.reset_index(drop=True)
    ann = gr.annotations.copy()
    if ann.index.name == "metabolite":
        ann = ann.reset_index()
    gr.annotations = ann.loc[:, ~ann.columns.duplicated()].drop_duplicates(
        subset=["metabolite"]
    ).reset_index(drop=True)
    return gr


def production_fluxes(results, atol=1e-3):
    ex = results.exchanges.reset_index(drop=True)
    pos = ex[(ex.taxon != "medium") & (ex.direction == "export")].copy()
    pos["w"] = pos.abundance * pos.flux.abs()
    rates = pos.groupby(["sample_id", "metabolite"], as_index=False)["w"].sum()
    return rates.rename(columns={"w": "flux"}).loc[lambda d: d.flux > atol]


def lasso_production_associations(results, phenotype, atol=1e-3, min_coef=0.001):
    exchanges = production_fluxes(results, atol=atol)
    exchanges = exchanges[exchanges.sample_id.isin(phenotype.index)]
    fluxes = exchanges.pivot_table(
        index="sample_id", columns="metabolite", values="flux", fill_value=atol
    ).map(np.log)
    meta = phenotype.loc[fluxes.index]
    fluxes = fluxes.loc[:, fluxes.std(axis=0) > 1e-6]
    X = StandardScaler().fit_transform(fluxes)
    cv = LogisticRegressionCV(
        penalty="l1", solver="liblinear", cv=2,
        Cs=np.power(10.0, np.arange(-6, 6, 0.5)), max_iter=50000,
    ).fit(X, meta)
    model = LogisticRegression(
        penalty="l1", solver="liblinear", C=cv.C_[0], max_iter=10000,
    ).fit(X, meta)
    coefs = pd.DataFrame({"metabolite": fluxes.columns, "coef": model.coef_[0]})
    return coefs.loc[coefs.coef.abs() >= min_coef].sort_values("coef")


def plot_lasso_coefs(coefs, title="LASSO coefficients"):
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(coefs))))
    ax.barh(coefs.metabolite, coefs.coef)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlabel("coefficient")
    ax.set_title(title)
    plt.tight_layout()
    return fig


growth_results = clean_growth_results(growth_results)


In [ ]:
from micom.viz import *

viz = plot_growth(growth_results)

Normally, we could call `viz.view()` afterwards and it would open it in our web browser. However, this will not work in Colab. However, the plot function create the file `growth_rates_[DATE].html` in your `materials` folder. To open it simply download that file and view it in your browser. We can see that there are many things going on, but it's not super clear. Let's continue.

## Growth niches

Two really important questions are 'what dietary nutrients are consumed by the microbiota and what metabolites do the microbiota produce?' We provided nutrients in our medium, but we don't actually know yet what was eaten by the microbiota. Let's check that out using the `plot_exchanges_per_sample` function.

In [ ]:
plot_exchanges_per_sample(growth_results)

We can have a look at the results after downloading `sample_exchanges_[DATE].html`. It would be even better if we could visualize which taxa compete for similar resources. We can create a niche plot by using `plot_exchanges_per_taxon`.

In [ ]:
plot_exchanges_per_taxon(growth_results, perplexity=4, direction="import")

## Metabolic connections to a phenotype

How do production fluxes relate to recurrent *C. diff* infection? Use `plot_association` ([docs](https://micom-dev.github.io/micom/viz.html)):

1. Community-wide production fluxes per metabolite
2. Non-parametric test per metabolite vs phenotype
3. FDR-corrected q-values
4. LASSO logistic regression for global prediction performance

With **8 samples**, strict FDR (`0.05`) often shows nothing — we use `fdr_threshold=0.5` for this demo. The matplotlib plot below shows the **legacy `plot_fit` LASSO coefficients** for comparison.


In [ ]:
from micom.viz import plot_association

manifest.index = manifest.sample_id
pheno = manifest.disease_stat

pl = plot_association(
    growth_results,
    pheno,
    variable_type="binary",
    variable_name="disease status",
    flux_type="production",
    atol=1e-3,
    fdr_threshold=0.5,
    filename="association_cdiff.html",
)
print("Saved:", pl.filename)

coefs = lasso_production_associations(growth_results, pheno, atol=1e-3)
plot_lasso_coefs(coefs, title="Legacy plot_fit-style LASSO coefficients")
plt.show()


Download `association_cdiff.html` from the file browser (folder icon, left sidebar).

- **Per-metabolite panel:** metabolites with q < `fdr_threshold`
- **Global panel:** cross-validated LASSO prediction

Negative statistics → higher production in recurrent *C. diff*; positive → healthy.

Interpret cautiously: n=8 is very small.


## Mock phenotype: propionate

From the [MICOM viz tutorial](https://micom-dev.github.io/micom/viz.html): phenotype = above-median propionate production.


In [ ]:
# Use production_fluxes helper (avoids production_rates pandas bug on MICOM ≥ 0.35)
prod = production_fluxes(growth_results, atol=1e-3)
propionate = (
    prod.loc[prod.metabolite == "ppa[e]", ["sample_id", "flux"]]
    .set_index("sample_id")["flux"]
)
high_propionate = propionate > propionate.median()

pl = plot_association(
    growth_results,
    high_propionate,
    variable_type="binary",
    variable_name="high propionate",
    flux_type="production",
    atol=1e-3,
    fdr_threshold=0.5,
    filename="association_propionate.html",
)
print("Saved:", pl.filename)


## Microbial interactions

`plot_focal_interactions` — exchanges for one focal taxon. `plot_mes` — Metabolic Exchange Score across groups.

Taxon names must match **genus-level** models (`g__...`).


In [ ]:
from micom.viz import plot_focal_interactions, plot_mes

print(growth_results.growth_rates.taxon.unique())

pl = plot_focal_interactions(
    growth_results,
    taxon="g__Akkermansia",
    filename="focal_akkermansia.html",
)
print("Saved:", pl.filename)

groups = manifest.disease_stat.copy()
groups.name = "disease status"

pl = plot_mes(growth_results, groups=groups, filename="mes_cdiff.html")
print("Saved:", pl.filename)


# 🏫 Exercises

Up to now, we have mostly used MICOM's "high-level" API, which is designed for working with several samples in parallel. However, MICOM also allows you to work with single models. We will choose a single sample now for further analysis.

First, let's recall what samples we had.

In [ ]:
manifest

Let's look further into the surprising *C. diff.* individual that looked very similar to the healthy subjects. We will apply the same diet as before.

In [ ]:
from micom import load_pickle
from micom.qiime_formats import load_qiime_medium

medium = load_qiime_medium("western_diet_gut.qza")
medium.index = medium.reaction

com = load_pickle("models/ERR1883248.pickle")
com.medium = medium.flux
com

This is a MICOM community object. MICOM community models are full [COBRApy](https://opencobra.github.io/cobrapy/) models (with sprinkles on top) and there is a whole bunch of stuff we could do with it.

## Microbe-microbe interactions

Let's dive a bit more into competition and cooperation between taxa. We can start by simulating taxa knockouts using `com.knockout_taxa`. For that we will remove each of the taxa from the model (one-at-a-time) and see how that affects the growth rates of all other taxa. If other taxa grow faster after the knockout, they were competing with the knocked-out taxon. However, if they grow slower, they were cooperating with the knocked-out taxon.

See the docs for more info: https://micom-dev.github.io/micom/taxa_knockouts.html.

Bonus points if you can visualize your results. How would you deal with vastly different scales of growth rates between taxa?

> Oh geez. I once saw somebody plot a heatmap with Seaborn using a Pandas DataFrame. I wish I could remember where that was... 🤔


In [ ]:
import seaborn as sns

ko = com.knockout_taxa(fraction=0.8, method="change")

sns.heatmap(ko, cmap="seismic", center=0)

## The flux capacitator

Here is how you would run the cooperative tradeoff for a single model. We can follow that up with pFBA to get all fluxes in the system.

In [ ]:
sol = com.cooperative_tradeoff(fraction=0.5, fluxes=True, pfba=True)
sol

The returned solution contains all fluxes in `sol.fluxes`. An `NaN` entry denotes that this reaction is not present in the organism.

In [ ]:
sol.fluxes.head()

The acetate export reaction has the name `EX_ac(e)`. Identify the primary acetate producer in the system. Don't forget about the accuracy of 10<sup>-3</sup>.

In [ ]:
sol.fluxes["EX_ac(e)"]

Look up at least one other reaction at https://www.vmh.life/#microbes/reactions, and get it's predicted fluxes. Does the reaction take place? If yes in which organism? Can you identify the most active fluxes in the community?

In [ ]:
sol.fluxes["EX_ac(e)"] * com.abundances

# 🔵 Addendum


## Choosing a tradeoff value

Even if you don't have growth rates available you can still use your data to choose a decent tradeoff value. This can be done by choosing the largest tradeoff value that still allows growth for the majority of the taxa that you observed in the sample (if they are present at an appreciable abundance, they should be able to grow). This can be done with the `tradeoff` workflow in MICOM that will run cooperative tradeoff with varying tradeoff values, which can be visualized with the `plot_tradeoff` function.

In [ ]:
from micom.workflows import tradeoff
import micom

tradeoff_results = tradeoff(manifest, "models", medium, threads=2)
tradeoff_results.to_csv("tradeoff.csv", index=False)

plot_tradeoff(tradeoff_results, tolerance=1e-4)

After opeing `tradeoff_[DATE].html` you will see that, for our example here, all tradeoff values work great. This is because we modeled very few taxa, which keeps the compettion down. If you would allow for fewer abundant taxa in the models, this would change drastically. For instance, here is an example from a colorectal cancer data set:

[![tradeoff example](https://micom-dev.github.io/micom/_images/tradeoff.png)](https://micom-dev.github.io/micom/_static/tradeoff.html)

You can see how not using the cooperative tradeoff would give you nonsense results where only 10% of all observed taxa grew. A tradeoff value of 0.6-0.8 would probably be a good choice for this particular data set.